# 02 · Análisis: el costo invisible del progreso
### ¿Cambia la identificación de los países que desacoplan su crecimiento de las emisiones de CO₂ al pasar de la contabilidad territorial a la basada en consumo?

**Proyecto 2 · DATA-1002 Aplicaciones en Ciencia de Datos · Universidad de los Andes**

Este notebook parte de `data/processed/datos_integrados.csv` (generado por `01_preparacion_datos.ipynb`) y sigue **el mismo método del dashboard** (`src/dashboard_desacoplamiento.py`): compara un año inicial con un año final, por defecto el primero y el último del panel (1990 y 2023). Así las cifras del notebook coinciden con las que muestra el dashboard al abrirse.

| Sección | Sesión del curso |
|---|---|
| 1. Comprensión del problema | 2 |
| 2. Carga de los datos integrados | 3 y 4 |
| 3. Análisis exploratorio | 6 y 7 |
| 4. Clasificación del desacoplamiento y respuesta a la pregunta | 6 y 7 |
| 5. Desagregación, robustez y sesgos | 7 y 11 |
| 6. Hallazgos y recomendaciones | 6, 7 y 14 |
| 7. Limitaciones y fuentes | 3 y 11 |

## 1. Comprensión del problema

| Pregunta | Respuesta |
|---|---|
| **¿Quién?** | Comité técnico de desarrollo sostenible y transición climática (usuario). Afecta a gobiernos y a la población de países que exportan o importan bienes intensivos en carbono. |
| **¿Qué?** | Saber qué países crecen sin aumentar sus emisiones de CO₂ y si ese resultado se sostiene al contar las emisiones del comercio internacional. |
| **¿Dónde?** | Países con datos de emisiones basadas en consumo (unos 120). |
| **¿Cuándo?** | 1990–2023, período común de las cuatro fuentes. |
| **¿Por qué?** | Con la contabilidad territorial, un país puede parecer exitoso aunque parte de su producción contaminante ocurra en otros países. |

**Objetivo de negocio:** apoyar al comité en la identificación de **países referentes** (su desacoplamiento se sostiene con ambas contabilidades) y de **países que requieren diagnóstico adicional** (su avance territorial no se sostiene al incluir el comercio).

**Pregunta analítica:** ¿En qué medida cambia la identificación de países que han logrado desacoplar su crecimiento económico de las emisiones de CO₂ al comparar las emisiones territoriales con las asociadas al consumo?

**Hipótesis:** una parte de los países con desacoplamiento absoluto territorial lo pierde con la contabilidad de consumo, y esto ocurre más en los **importadores netos** de emisiones.

**Alcance:** descriptivo. No se busca un modelo predictivo ni establecer causalidad (lo confirmó el profesor en la reunión del 23/11).

## 2. Carga de los datos integrados

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path(".") if Path("data").exists() else Path("..")
FIGURAS = RAIZ / "outputs" / "figures"
FIGURAS.mkdir(parents=True, exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

df = pd.read_csv(RAIZ / "data" / "processed" / "datos_integrados.csv")

assert not df.duplicated(["codigo", "anio"]).any()
print("Filas (país-año):", len(df), "| Países:", df["codigo"].nunique(), "| Años:", df["anio"].min(), "–", df["anio"].max())
df.head()

| Columna | Significado | Unidad |
|---|---|---|
| `pib` | PIB real, PPA | dólares internacionales de 2021 |
| `co2_territorial`, `co2_consumo` | Emisiones anuales de CO₂ | toneladas |
| `poblacion` | Población | personas |
| `territorial_per_capita`, `consumo_per_capita` | CO₂ por persona | t CO₂ por persona |
| `intensidad_territorial`, `intensidad_consumo` | CO₂ por cada 1.000 dólares de PIB | t CO₂ / 1.000 dólares |
| `brecha_emisiones` | consumo − territorial | t CO₂ (positivo = importador neto de emisiones) |

## 3. Análisis exploratorio

### 3.1 ¿Cuánto pesa el comercio en las emisiones? (2023)

In [ ]:
ultimo = df[df["anio"] == 2023].copy()
ultimo["brecha_pct"] = ultimo["brecha_emisiones"] / ultimo["co2_territorial"] * 100
ultimo["balance_comercio"] = np.where(ultimo["brecha_emisiones"] > 0, "Importador neto", "Exportador neto")

print(ultimo[["territorial_per_capita", "consumo_per_capita", "intensidad_territorial", "brecha_pct"]].describe().round(2).to_string())
print()
print(ultimo["balance_comercio"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(ultimo["brecha_pct"].clip(-100, 200), bins=30, color="#2a9d8f", edgecolor="white")
ax.axvline(0, color="black", lw=1)
ax.axvline(ultimo["brecha_pct"].median(), color="#e76f51", ls="--", label=f"Mediana: {ultimo['brecha_pct'].median():.0f} %")
ax.set_title("En 2023, la mayoría de países consume más CO₂ del que produce en su territorio")
ax.set_xlabel("Brecha consumo − territorial (% de las emisiones territoriales; recortada a [−100, 200])")
ax.set_ylabel("Número de países")
ax.legend()
plt.tight_layout()
fig.savefig(FIGURAS / "01_brecha_2023.png", dpi=150)
plt.show()

**Lectura:** unos tres de cada cuatro países son importadores netos de emisiones (consumo > territorial). Como la distribución es asimétrica, se reporta la mediana además de la media.

## 4. Clasificación del desacoplamiento (mismo método del dashboard)

Para cada país se calcula la variación porcentual del PIB y de las emisiones entre el **año inicial** y el **año final**, y se clasifica:

| Clase | Regla |
|---|---|
| Absoluto | El PIB crece y las emisiones caen |
| Relativo | El PIB crece y las emisiones crecen menos que el PIB |
| Sin desacoplamiento | El PIB crece y las emisiones crecen igual o más que el PIB |
| Sin crecimiento | El PIB no creció |

Solo se comparan países con datos en ambos años y valores positivos en el año inicial. La función `resumen_periodo` reproduce el cálculo del dashboard.

In [ ]:
def clasificacion(pib, emisiones):
    if pib <= 0:
        return "Sin crecimiento"
    if emisiones < 0:
        return "Absoluto"
    if emisiones < pib:
        return "Relativo"
    return "Sin desacoplamiento"


def resumen_periodo(datos, anio_inicio, anio_final):
    """Mismo cálculo del dashboard: compara el año inicial con el año final para cada país."""
    df_inicio = datos[datos.anio == anio_inicio]
    df_fin = datos[datos.anio == anio_final]
    resumen = df_inicio.merge(df_fin, on="codigo", how="inner", suffixes=("_inicio", "_fin"), validate="1:1")
    resumen["pais"] = resumen.pais_inicio
    resumen = resumen[(resumen.pib_inicio > 0) & (resumen.co2_territorial_inicio > 0) &
                      (resumen.co2_consumo_inicio > 0)].copy()

    variacion = lambda col: (resumen[f"{col}_fin"] / resumen[f"{col}_inicio"] - 1) * 100
    resumen["var_pib"] = variacion("pib")
    resumen["var_territorial"] = variacion("co2_territorial")
    resumen["var_consumo"] = variacion("co2_consumo")
    resumen["var_territorial_pc"] = variacion("territorial_per_capita")
    resumen["var_consumo_pc"] = variacion("consumo_per_capita")
    resumen["var_intensidad_territorial"] = variacion("intensidad_territorial")
    resumen["var_intensidad_consumo"] = variacion("intensidad_consumo")
    resumen["brecha_pp"] = resumen.var_consumo - resumen.var_territorial

    resumen["Territorial"] = [clasificacion(p, e) for p, e in zip(resumen.var_pib, resumen.var_territorial)]
    resumen["Consumo"] = [clasificacion(p, e) for p, e in zip(resumen.var_pib, resumen.var_consumo)]
    resumen["cambia"] = resumen.Territorial != resumen.Consumo
    return resumen.set_index("codigo")


ANIO_INICIO, ANIO_FINAL = df.anio.min(), df.anio.max()   # valores por defecto del dashboard
ORDEN = ["Absoluto", "Relativo", "Sin desacoplamiento", "Sin crecimiento"]

resumen = resumen_periodo(df, ANIO_INICIO, ANIO_FINAL)

print(f"Período: {ANIO_INICIO} frente a {ANIO_FINAL}")
print("Países comparables:", len(resumen))
print("Sin datos en ambos años:", sorted(set(df.codigo) - set(resumen.index)))

In [ ]:
conteo = pd.DataFrame({
    "Territorial": resumen["Territorial"].value_counts(),
    "Consumo": resumen["Consumo"].value_counts(),
}).reindex(ORDEN).fillna(0).astype(int)
conteo.loc["Total"] = conteo.sum()
conteo

### 4.1 Respuesta a la pregunta: ¿cuántos países cambian de clase?

In [ ]:
cruce = pd.crosstab(resumen["Territorial"], resumen["Consumo"],
                    rownames=["Clase territorial"], colnames=["Clase de consumo"])
cruce = cruce.reindex(index=ORDEN, columns=ORDEN, fill_value=0)

n_cambian = int(resumen["cambia"].sum())
abs_terr = int((resumen["Territorial"] == "Absoluto").sum())
abs_cons = int((resumen["Consumo"] == "Absoluto").sum())
porcentaje_cambio = n_cambian / len(resumen) * 100

print(f"Cambian de clase al usar consumo: {n_cambian} de {len(resumen)} ({porcentaje_cambio:.1f} %)")
print(f"Desacoplamiento absoluto: {abs_terr} con territorial → {abs_cons} con consumo")
cruce

In [ ]:
m = cruce.values
fig, ax = plt.subplots(figsize=(7.5, 6))
ax.imshow(m, cmap="Blues")
ax.set_xticks(range(4), ORDEN, rotation=15)
ax.set_yticks(range(4), ORDEN)
for i in range(4):
    for j in range(4):
        ax.text(j, i, m[i, j], ha="center", va="center", fontsize=13, color="white" if m[i, j] > m.max() / 2 else "black")
ax.set_xlabel("Clase con emisiones de consumo")
ax.set_ylabel("Clase con emisiones territoriales")
ax.set_title(f"{n_cambian} de {len(resumen)} países cambian de clase al contar el comercio\n"
             f"({ANIO_INICIO} frente a {ANIO_FINAL}; fuera de la diagonal = cambia)")
plt.tight_layout()
fig.savefig(FIGURAS / "02_tabla_cruzada.png", dpi=150)
plt.show()

### 4.2 Casos para el comité

- **Referentes:** desacoplamiento absoluto con ambas contabilidades.
- **Requieren diagnóstico:** desacoplamiento absoluto territorial que no se sostiene con consumo.

In [ ]:
cols = ["pais", "var_pib", "var_territorial", "var_consumo", "Territorial", "Consumo"]
referentes = resumen[(resumen.Territorial == "Absoluto") & (resumen.Consumo == "Absoluto")]
diagnostico = resumen[(resumen.Territorial == "Absoluto") & (resumen.Consumo != "Absoluto")]
ganan = resumen[(resumen.Territorial != "Absoluto") & (resumen.Consumo == "Absoluto")]

print(f"Referentes: {len(referentes)} | Requieren diagnóstico: {len(diagnostico)} | "
      f"Ganan el absoluto con consumo: {len(ganan)} ({', '.join(ganan.pais)})")
diagnostico[cols].sort_values("var_consumo", ascending=False)

In [ ]:
referentes[cols].sort_values("var_consumo")

**Cuidado con Bélgica:** su CO₂ de consumo de 1990 es una copia del territorial (ver `01_preparacion_datos.ipynb`), así que su aparición entre los casos de diagnóstico puede deberse a ese dato y no a un cambio real (ver robustez en la sección 5.3).

### 4.3 Relación entre ambas contabilidades

Se grafica el cociente *año final / año inicial* en escala logarítmica, para ver en un mismo gráfico a países que multiplicaron sus emisiones por 40 y a países que las redujeron. Un valor de 1 significa que no cambiaron.

In [ ]:
print(f"Correlación entre la variación territorial y la de consumo: "
      f"Spearman {resumen[['var_territorial', 'var_consumo']].corr(method='spearman').iloc[0, 1]:.2f} | "
      f"Pearson {resumen[['var_territorial', 'var_consumo']].corr().iloc[0, 1]:.2f}")

colores = {"Absoluto": "seagreen", "Relativo": "steelblue", "Sin desacoplamiento": "darkorange", "Sin crecimiento": "gray"}
rt = 1 + resumen["var_territorial"] / 100
rc = 1 + resumen["var_consumo"] / 100
lim = [min(rt.min(), rc.min()) * 0.8, max(rt.max(), rc.max()) * 1.25]

fig, ax = plt.subplots(figsize=(8.5, 7))
for clase, idx in resumen.groupby("Territorial").groups.items():
    ax.scatter(rt[idx], rc[idx], s=32, alpha=0.85, color=colores[clase], label=f"{clase} (territorial)")
ax.plot(lim, lim, color="grey", lw=1, ls="--", label="Igual cambio en ambas")
ax.axhline(1, color="black", lw=0.8)
ax.axvline(1, color="black", lw=0.8)
for codigo in diagnostico.index:
    ax.annotate(codigo, (rt[codigo], rc[codigo]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel(f"CO₂ territorial: {ANIO_FINAL} / {ANIO_INICIO} (escala log)")
ax.set_ylabel(f"CO₂ de consumo: {ANIO_FINAL} / {ANIO_INICIO} (escala log)")
ax.set_title("Los países que reducen su CO₂ territorial no siempre reducen el de consumo\n"
             "(arriba de la diagonal: el consumo creció más que la producción doméstica)")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
fig.savefig(FIGURAS / "03_territorial_vs_consumo.png", dpi=150)
plt.show()

**Lectura:** ambas contabilidades están muy correlacionadas (Spearman ≈ 0,9); en la mayoría de países cuentan la misma historia. Las diferencias están en el cuadrante superior izquierdo (etiquetados): países que redujeron su CO₂ territorial mientras el de consumo creció.

### 4.4 Valores atípicos (IQR)

In [ ]:
def atipicos_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return serie[(serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)]

for col in ["var_pib", "var_territorial", "var_consumo"]:
    at = atipicos_iqr(resumen[col]).sort_values(ascending=False)
    print(f"{col}: {len(at)} atípicos → " + ", ".join(f"{i} ({v:,.0f} %)" for i, v in at.items()))

**Decisión:** son casos reales, no errores: economías de ingreso bajo que partieron de emisiones muy bajas en 1990 o que crecieron muy rápido. **Se conservan**: la clasificación usa umbrales (0 y el crecimiento del PIB), no promedios, así que no la distorsionan. Por eso el gráfico usa escala logarítmica.

### 4.5 Trayectorias por país

Como en la última sección del dashboard: índice base 100 en el año inicial. Además, siguiendo la sugerencia del profesor, se sombrean los años en que el CO₂ territorial creció más que el PIB respecto al año anterior.

In [ ]:
def anios_sin_desacople(serie, columna_co2):
    crec_pib = serie["pib"].pct_change() * 100
    crec_co2 = serie[columna_co2].pct_change() * 100
    return serie.loc[(crec_co2 > 0) & (crec_co2 > crec_pib), "anio"].tolist()

def trayectoria(codigo, ax):
    s = df[df["codigo"] == codigo].sort_values("anio").reset_index(drop=True)
    base = s.iloc[0]
    for anio in anios_sin_desacople(s, "co2_territorial"):
        ax.axvspan(anio - 0.5, anio + 0.5, color="darkorange", alpha=0.12, lw=0)
    ax.plot(s.anio, s.pib / base.pib * 100, label="PIB real", color="navy", lw=2)
    ax.plot(s.anio, s.co2_territorial / base.co2_territorial * 100, label="CO₂ territorial", color="steelblue")
    ax.plot(s.anio, s.co2_consumo / base.co2_consumo * 100, label="CO₂ consumo", color="darkorange", ls="--")
    ax.axhline(100, color="grey", lw=0.6)
    ax.set_title(s.pais.iloc[0])

ejemplos = ["GBR", "CHE", "CHN", "COL"]   # un referente, un caso de diagnóstico y dos contrastes
fig, axes = plt.subplots(1, len(ejemplos), figsize=(16, 4))
for c, ax in zip(ejemplos, axes):
    trayectoria(c, ax)
axes[0].set_ylabel(f"Índice ({ANIO_INICIO} = 100)")
axes[0].legend(fontsize=8)
fig.suptitle("PIB frente a CO₂ (sombreado: años en que el CO₂ territorial creció más que el PIB)", y=1.03)
plt.tight_layout()
fig.savefig(FIGURAS / "04_trayectorias.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Desagregación, robustez y sesgos

### 5.1 ¿Cambia el resultado al desagregar? (Sesiones 7 y 11)

Grupos de ingreso: terciles del PIB per cápita en el año final. Balance comercial de emisiones: año final.

In [ ]:
desag = resumen.assign(
    grupo_ingreso=pd.qcut(resumen.pib_fin / resumen.poblacion_fin, 3, labels=["Ingreso bajo", "Ingreso medio", "Ingreso alto"]),
    balance_final=np.where(resumen.brecha_emisiones_fin > 0, "Importador neto", "Exportador neto"),
)

def resumen_grupo(g):
    return pd.Series({
        "países": len(g),
        "cambian de clase": int(g["cambia"].sum()),
        "% que cambia": g["cambia"].mean() * 100,
        "absoluto territorial": int((g["Territorial"] == "Absoluto").sum()),
        "absoluto consumo": int((g["Consumo"] == "Absoluto").sum()),
    })

columnas = ["cambia", "Territorial", "Consumo"]
print("Por grupo de ingreso:")
print(desag.groupby("grupo_ingreso", observed=True)[columnas].apply(resumen_grupo).round(1).to_string())
print("\nPor balance de emisiones en el año final:")
print(desag.groupby("balance_final")[columnas].apply(resumen_grupo).round(1).to_string())

**Lectura:** el desacoplamiento absoluto se concentra en los países de ingreso alto, y es ahí donde contar el comercio "quita" referentes. En los países de ingreso bajo y medio el porcentaje que cambia de clase es mayor, pero sobre todo entre *relativo* y *sin desacoplamiento*.

**Hipótesis:** se cumple en parte. Entre los importadores netos, el desacoplamiento absoluto baja de forma clara al contar el comercio; entre los exportadores netos casi no cambia. Pero el porcentaje que cambia de clase en general es parecido en ambos grupos.

### 5.2 Per cápita e intensidad

Como en el selector del dashboard ("Emisiones per cápita" / "Intensidad de emisiones").

In [ ]:
var_pib_pc = ((resumen.pib_fin / resumen.poblacion_fin) / (resumen.pib_inicio / resumen.poblacion_inicio) - 1) * 100
clase_pc_terr = [clasificacion(p, e) for p, e in zip(var_pib_pc, resumen.var_territorial_pc)]
clase_pc_cons = [clasificacion(p, e) for p, e in zip(var_pib_pc, resumen.var_consumo_pc)]

pd.DataFrame({
    "Totales · territorial": resumen.Territorial.value_counts(),
    "Totales · consumo": resumen.Consumo.value_counts(),
    "Per cápita · territorial": pd.Series(clase_pc_terr).value_counts(),
    "Per cápita · consumo": pd.Series(clase_pc_cons).value_counts(),
}).reindex(ORDEN).fillna(0).astype(int)

In [ ]:
print("Países cuya intensidad territorial cayó:", int((resumen.var_intensidad_territorial < 0).sum()), "de", len(resumen))
print("Países cuya intensidad de consumo cayó:", int((resumen.var_intensidad_consumo < 0).sum()), "de", len(resumen))
print(f"Mediana del cambio de intensidad: territorial {resumen.var_intensidad_territorial.median():.1f} % | "
      f"consumo {resumen.var_intensidad_consumo.median():.1f} %")

### 5.3 Robustez

En el dashboard se pueden elegir otros años. Aquí se comparan algunos, y también el resultado sin los cinco países cuyo CO₂ de consumo de los años 90 es copia del territorial (Bélgica, Luxemburgo, Sudáfrica, Namibia y Botsuana).

In [ ]:
COPIADOS = ["BEL", "LUX", "ZAF", "NAM", "BWA"]

def resumen_corto(r, nombre):
    return {
        "período": nombre,
        "países": len(r),
        "absoluto territorial": int((r.Territorial == "Absoluto").sum()),
        "absoluto consumo": int((r.Consumo == "Absoluto").sum()),
        "cambian de clase": int(r.cambia.sum()),
        "% que cambia": round(r.cambia.mean() * 100, 1),
    }

pd.DataFrame([
    resumen_corto(resumen, f"{ANIO_INICIO} vs {ANIO_FINAL} (dashboard por defecto)"),
    resumen_corto(resumen[~resumen.index.isin(COPIADOS)], f"{ANIO_INICIO} vs {ANIO_FINAL} sin países con consumo copiado"),
    resumen_corto(resumen_periodo(df, 1995, ANIO_FINAL), f"1995 vs {ANIO_FINAL}"),
    resumen_corto(resumen_periodo(df, 2005, ANIO_FINAL), f"2005 vs {ANIO_FINAL}"),
]).set_index("período")

**Lectura:** con otros años o sin los países con datos copiados, entre el 19 % y el 22 % de los países cambia de clase, y siempre hay menos casos de desacoplamiento absoluto con la contabilidad de consumo. La conclusión no depende del período elegido.

### 5.4 Sesgos y uso responsable (Sesión 11)

In [ ]:
pib_mundo = pd.read_csv(RAIZ / "data" / "raw" / "gdp-worldbank.csv")
pib_mundo = pib_mundo[pib_mundo["Code"].str.fullmatch("[A-Z]{3}", na=False) & (pib_mundo["Year"] == ANIO_FINAL)]
cobertura = pib_mundo.loc[pib_mundo["Code"].isin(resumen.index), "GDP"].sum() / pib_mundo["GDP"].sum()

print(f"Países con PIB en {ANIO_FINAL}: {len(pib_mundo)} | comparables: {len(resumen)}")
print(f"Participación de los países comparables en el PIB mundial de {ANIO_FINAL}: {cobertura:.1%}")
fuera = pib_mundo[~pib_mundo["Code"].isin(resumen.index)].nlargest(10, "GDP")
print("Excluidos con mayor PIB:", ", ".join(fuera["Entity"]))

| Pregunta de auditoría | Respuesta |
|---|---|
| ¿Quiénes integran la población de interés? | Todos los países que buscan crecer reduciendo emisiones. |
| ¿Quiénes pueden entrar a la fuente? | Solo países con datos de comercio suficientes para estimar emisiones de consumo (~120). |
| ¿Quiénes están subrepresentados? | Países pequeños y de bajos ingresos, sobre todo de África y Asia Central. **Sesgo de selección.** |
| ¿La variable mide lo mismo para todos? | No del todo: el consumo se estima con modelos de comercio y en cinco países los años 90 son copia del territorial. **Sesgo de medición.** |
| ¿Hay faltantes sistemáticos o duplicaciones? | Los faltantes de consumo se concentran en países pobres; no hay duplicados. |
| ¿Cómo cambia el resultado al desagregar? | Mucho: el desacoplamiento absoluto es casi exclusivo del ingreso alto (5.1). |
| ¿Qué decisión puede afectar a quién? | Señalar a un país como referente o en diagnóstico puede afectar su reputación. Se presentan como **casos para estudiar**, no como juicios, y no se interpreta como fracaso que países pobres aumenten emisiones desde niveles muy bajos. |

## 6. Hallazgos y recomendaciones

In [ ]:
print(f"1. Cambian de clase: {n_cambian} de {len(resumen)} países ({porcentaje_cambio:.1f} %)")
print(f"2. Desacoplamiento absoluto: {abs_terr} (territorial) → {abs_cons} (consumo); referentes: {len(referentes)}")
print(f"3. Requieren diagnóstico: {len(diagnostico)} → {', '.join(diagnostico.pais)}")
print(f"4. Cobertura: {len(resumen)} países, {cobertura:.0%} del PIB mundial de {ANIO_FINAL}")

resumen.reset_index()[["codigo"] + cols + ["var_territorial_pc", "var_consumo_pc", "var_intensidad_territorial",
                                          "var_intensidad_consumo", "brecha_pp", "cambia"]] \
    .to_csv(RAIZ / "data" / "processed" / "clasificacion_paises.csv", index=False)
print("\nGuardado: data/processed/clasificacion_paises.csv")

**Hallazgo 1. Uno de cada cinco países cambia de clase al contar el comercio.** 26 de 116 países (22,4 %) tienen una clase distinta con emisiones de consumo entre 1990 y 2023; con otros años el porcentaje es similar (19 %–22 %). La contabilidad territorial por sí sola no basta para identificar desacoplamiento.

**Hallazgo 2. Contar el comercio reduce los casos de desacoplamiento absoluto:** 38 con la contabilidad territorial frente a 30 con la de consumo; 28 lo logran con ambas (referentes, en su mayoría europeos de ingreso alto).

**Hallazgo 3. Diez países requieren diagnóstico adicional:** reducen su CO₂ territorial mientras el de consumo crece (por ejemplo Suiza, Estados Unidos, Malta). Es compatible con trasladar producción al exterior, pero los datos no lo demuestran. En el caso de Bélgica puede influir el dato copiado de 1990.

**Hallazgo 4. El desacoplamiento absoluto se concentra en el ingreso alto:** 25 de 39 países de ingreso alto lo logran con la contabilidad territorial, frente a 3 de 39 de ingreso bajo.

**Recomendaciones para el comité:**
1. Evaluar el desacoplamiento con ambas contabilidades, no solo con la territorial.
2. Estudiar los referentes como posibles casos de política climática exitosa.
3. Pedir un diagnóstico de los países cuyo avance territorial no se sostiene con consumo, empezando por el origen de sus importaciones de emisiones.
4. No comparar a países de ingreso bajo con los de ingreso alto con la misma vara: parten de emisiones per cápita muy bajas.

## 7. Limitaciones y fuentes

**Limitaciones**
1. **Cobertura:** solo ~120 países tienen emisiones de consumo; quedan fuera muchos países pequeños y de bajos ingresos. Taiwán y Venezuela quedan fuera por falta de PIB.
2. **Alcance de las emisiones:** solo CO₂ fósil e industrial; no incluye cambio de uso del suelo, aviación y navegación internacional, ni otros gases.
3. **Consumo estimado:** proviene de modelos de comercio. En Bélgica, Luxemburgo, Sudáfrica, Namibia y Botsuana los años 90 son copia del territorial.
4. **Años puntuales:** la comparación usa dos años; un año atípico al inicio o al final puede mover la clasificación (ver 5.3).
5. **Sin causalidad:** el análisis es descriptivo.

**Fuentes (licencia CC BY 4.0):**
- Global Carbon Budget (2026) – with major processing by Our World in Data. https://ourworldindata.org/co2-emissions
- Eurostat, OECD, IMF, and World Bank (2026) – with minor processing by Our World in Data. https://ourworldindata.org/grapher/gdp-worldbank
- HYDE (2023); Gapminder (2022); UN WPP (2024) – with major processing by Our World in Data. https://ourworldindata.org/grapher/population